# 23-20 · Тестируем перемещение и отмену

Практика к разделу [«Проверяем перемещение и отмену»](../../site/chapters/glava-23/23-25-testy-peremeshheniya.html). Использует настоящий пакет `safesort`.

## Цель

Написать и запустить три теста в духе тех, что живут в `projects/python/safesort/tests/test_executor.py` и `test_manifest.py`: успешное перемещение, полная отмена и конфликт при восстановлении.

## Рабочий пример — тесты apply_plan() и undo()

In [1]:
import tempfile
from pathlib import Path

from safesort.config import Config
from safesort.scanner import scan
from safesort.planner import build_plan
from safesort.executor import apply_plan
from safesort.manifest import write_manifest, undo
from safesort.models import MoveOperation, SortPlan


def test_apply_plan_moves_file_to_destination(tmp_path):
    source = tmp_path / "otchet.pdf"
    source.write_text("...", encoding="utf-8")
    destination = tmp_path / "Sorted" / "documents" / "otchet.pdf"
    plan = SortPlan(root=tmp_path, operations=(MoveOperation(source, destination),))

    results = apply_plan(plan)

    assert results[0].completed is True
    assert not source.exists()
    assert destination.exists()


def test_undo_restores_original_location(tmp_path):
    source = tmp_path / "otchet.pdf"
    source.write_text("...", encoding="utf-8")
    nastrojki = Config()
    plan = build_plan(scan(tmp_path, nastrojki), tmp_path, nastrojki)
    moves = apply_plan(plan)
    manifest_obj, _ = write_manifest(tmp_path, moves)

    result = undo(manifest_obj)

    assert source.exists()
    assert result.conflicts == ()


def test_undo_refuses_to_overwrite_conflict(tmp_path):
    source = tmp_path / "otchet.pdf"
    source.write_text("оригинал", encoding="utf-8")
    nastrojki = Config()
    plan = build_plan(scan(tmp_path, nastrojki), tmp_path, nastrojki)
    moves = apply_plan(plan)
    manifest_obj, _ = write_manifest(tmp_path, moves)

    source.write_text("кто-то создал новый файл здесь", encoding="utf-8")
    result = undo(manifest_obj)

    assert len(result.conflicts) == 1
    assert source.read_text(encoding="utf-8") == "кто-то создал новый файл здесь"


for test_func in (
    test_apply_plan_moves_file_to_destination,
    test_undo_restores_original_location,
    test_undo_refuses_to_overwrite_conflict,
):
    with tempfile.TemporaryDirectory() as tmp:
        test_func(Path(tmp))
    print(f"OK: {test_func.__name__}")

Refusing to undo /tmp/tmpi7f3bdm4/Sorted/documents/otchet.pdf -> /tmp/tmpi7f3bdm4/otchet.pdf: a file already exists at the original location: /tmp/tmpi7f3bdm4/otchet.pdf


OK: test_apply_plan_moves_file_to_destination
OK: test_undo_restores_original_location
OK: test_undo_refuses_to_overwrite_conflict


## Задание ★★ Самостоятельная задача

Напишите `test_apply_plan_reports_missing_source(tmp_path)`, которая планирует перемещение уже не существующего файла и проверяет, что `completed` равен `False`.

In [2]:
def test_apply_plan_reports_missing_source(tmp_path):
    source = tmp_path / "prizrak.pdf"  # файла никогда не было
    destination = tmp_path / "Sorted" / "documents" / "prizrak.pdf"
    plan = SortPlan(root=tmp_path, operations=(MoveOperation(source, destination),))

    results = apply_plan(plan)

    assert results[0].completed is False
    assert results[0].error is not None


with tempfile.TemporaryDirectory() as tmp:
    test_apply_plan_reports_missing_source(Path(tmp))
print("OK: test_apply_plan_reports_missing_source")

Source vanished before it could be moved: /tmp/tmp3ldl6lel/prizrak.pdf ([Errno 2] No such file or directory: '/tmp/tmp3ldl6lel/prizrak.pdf')


OK: test_apply_plan_reports_missing_source
